# Chapter 7: Multiple Regression Analysis with Qualitative Regressors

Topics: dummy variables, boolean variables, categorical variables, ANOVA, interaction with dummies.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('wage1')

## 7.1 Dummy Variables as Regressors

## Step 2 – Dummy (Binary) Variables

A **dummy variable** $d_i \in \{0,1\}$ captures group membership.

**Dummy intercept shift model:**
$$y_i = \beta_0 + \delta_0 d_i + \beta_1 x_i + u_i$$

- Group $d=0$ (base): $E(y|x,d=0) = \beta_0 + \beta_1 x$
- Group $d=1$: $E(y|x,d=1) = (\beta_0+\delta_0) + \beta_1 x$

So $\delta_0$ is the **intercept difference** between the two groups, holding $x$ constant (*ceteris paribus*).

> **Dummy variable trap:** with $m$ categories, include only $m-1$ dummies (omit one as the base/reference group) to avoid perfect multicollinearity.

In [ ]:
# Gender wage gap
model = smf.ols('wage ~ educ + exper + tenure + female', data=df).fit()
print(model.summary())
print(f"\nGender gap: women earn ${abs(model.params['female']):.2f}/hr less, ceteris paribus")

## 7.2 Boolean Variables

## Step 3 – Interacting Dummies with Continuous Variables

To allow **different slopes** across groups:
$$y_i = \beta_0 + \delta_0 d_i + \beta_1 x_i + \delta_1 (d_i \cdot x_i) + u_i$$

- Group $d=0$: slope = $\beta_1$
- Group $d=1$: slope = $\beta_1 + \delta_1$

**Example – gender wage gap with different returns to education:**
$$\log(wage_i) = \beta_0 + \delta_0 female_i + \beta_1 educ_i + \delta_1 (female_i \cdot educ_i) + u_i$$

In [ ]:
df['married'] = df['married'].astype(bool)
df['female']  = df['female'].astype(bool)

m_bool = smf.ols('wage ~ educ + exper + tenure + married + female', data=df).fit()
print(m_bool.summary())

## 7.3 Categorical Variables and ANOVA

## Step 4 – Categorical Variables and ANOVA

For a categorical variable with $m$ levels, create $m-1$ dummy variables.

In statsmodels, use `C(variable)` in the formula to handle this automatically.

**Chow test** – tests whether the same regression applies to two groups:
$$F = \frac{(SSR_r - SSR_{ur})/k}{SSR_{ur}/(n-2k)} \sim F_{k,\,n-2k}$$

where $SSR_r$ is from the pooled regression, $SSR_{ur} = SSR_1 + SSR_2$ from separate group regressions, and $k$ = number of parameters in each group regression.

In [ ]:
# Use number of dependents as categorical
m_cat = smf.ols('wage ~ educ + exper + C(numdep)', data=df).fit()
print(m_cat.summary())

# ANOVA
anova_table = sm.stats.anova_lm(m_cat, typ=2)
print("\nANOVA Table:")
print(anova_table)

## 7.5 Interactions and Differences Across Groups

In [ ]:
# Chow test approach: separate slopes by gender
m_gender = smf.ols('wage ~ educ * female + exper * female + tenure', data=df).fit()
print(m_gender.summary())

# Plot wage-education line by gender
educ_range = np.linspace(0, 18, 100)
b = m_gender.params
wage_men   = b['Intercept'] + b['educ'] * educ_range
wage_women = (b['Intercept'] + b['female']) + (b['educ'] + b['educ:female']) * educ_range

plt.figure(figsize=(8, 5))
plt.plot(educ_range, wage_men,   label='Men',   color='steelblue', lw=2)
plt.plot(educ_range, wage_women, label='Women', color='darkorange', lw=2)
plt.xlabel('Education'); plt.ylabel('Predicted Wage')
plt.title('Wage-Education Relationship by Gender')
plt.legend()
plt.savefig(f"{output_dir}/fig1_gender_wage_educ.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_07_Qualitative_Regressors_01.png'), dpi=150, bbox_inches='tight')
plt.show()